<a href="https://colab.research.google.com/github/kartikmogalpalli1993/Python/blob/main/Day16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

!pip install pyspark
!pip install great_expectations
!pip install pandas
!pip install matplotlib
!pip install reportlab

In [10]:
# ============================================================
# END TO END DATA QUALITY FRAMEWORK
# TECHNOLOGIES USED:
#   ✅ Python
#   ✅ PySpark
#   ✅ Data Quality Validation
#   ✅ Matplotlib Visualization
#   ✅ Automated PDF Report using ReportLab
# ============================================================

# ============================================================
# INSTALL REQUIRED PACKAGES
# RUN ONLY ONCE
# ============================================================

# !pip install pyspark pandas matplotlib reportlab

# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

import pandas as pd
import matplotlib.pyplot as plt

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

import os

# ============================================================
# CREATE SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("Swiggy_Zomato_Data_Quality_Framework") \
    .master("local[*]") \
    .getOrCreate()

print("✅ Spark Session Created")

# ============================================================
# READ CSV FILE
# ============================================================

# Replace with your actual file path

file_path = "swiggy_vs_zomato_3000.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("✅ CSV File Loaded Successfully")

# ============================================================
# DATASET SUMMARY
# ============================================================

row_count = df.count()
column_count = len(df.columns)

print("================================================")
print("DATASET SUMMARY")
print("================================================")

print(f"Total Rows    : {row_count}")
print(f"Total Columns : {column_count}")

# ============================================================
# DISPLAY SAMPLE DATA
# ============================================================

print("================================================")
print("SAMPLE DATA")
print("================================================")

df.show(5)

# ============================================================
# DISPLAY SCHEMA
# ============================================================

print("================================================")
print("SCHEMA INFORMATION")
print("================================================")

df.printSchema()

# ============================================================
# NULL VALUE ANALYSIS
# ============================================================

print("================================================")
print("NULL VALUE ANALYSIS")
print("================================================")

null_counts = []

for column_name in df.columns:

    null_count = df.filter(
        col(column_name).isNull()
    ).count()

    null_counts.append(
        (column_name, null_count)
    )

null_df = pd.DataFrame(
    null_counts,
    columns=["Column", "Null_Count"]
)

print(null_df)

# ============================================================
# DUPLICATE ANALYSIS
# ============================================================

duplicate_count = row_count - df.dropDuplicates().count()

print("================================================")
print("DUPLICATE ANALYSIS")
print("================================================")

print(f"Duplicate Rows : {duplicate_count}")

# ============================================================
# CONVERT TO PANDAS DATAFRAME
# ============================================================

pdf = df.toPandas()

# ============================================================
# DATA QUALITY VALIDATION
# ============================================================

print("================================================")
print("RUNNING DATA QUALITY VALIDATION")
print("================================================")

validation_results = []

# ------------------------------------------------------------
# VALIDATION 1 : ROW COUNT CHECK
# ------------------------------------------------------------

row_validation = row_count > 0

validation_results.append([
    "Row Count Validation",
    row_validation
])

# ------------------------------------------------------------
# VALIDATION 2 : NOT NULL CHECK
# ------------------------------------------------------------

first_column = pdf.columns[0]

not_null_validation = pdf[first_column].notnull().all()

validation_results.append([
    f"{first_column} Not Null Validation",
    not_null_validation
])

# ------------------------------------------------------------
# VALIDATION 3 : COLUMN STRUCTURE CHECK
# ------------------------------------------------------------

column_validation = len(pdf.columns) > 0

validation_results.append([
    "Column Structure Validation",
    column_validation
])

# ------------------------------------------------------------
# VALIDATION 4 : DUPLICATE CHECK
# ------------------------------------------------------------

duplicate_validation = duplicate_count == 0

validation_results.append([
    "Duplicate Validation",
    duplicate_validation
])

# ============================================================
# VALIDATION RESULTS
# ============================================================

validation_df = pd.DataFrame(
    validation_results,
    columns=["Validation", "Status"]
)

print("================================================")
print("VALIDATION RESULTS")
print("================================================")

print(validation_df)

# ============================================================
# GENERATE VISUALIZATION GRAPHS
# ============================================================

print("================================================")
print("GENERATING VISUALIZATION")
print("================================================")

graph_paths = []

# ------------------------------------------------------------
# IDENTIFY FIRST CATEGORICAL COLUMN
# ------------------------------------------------------------

categorical_column = None

for c in pdf.columns:

    if pdf[c].dtype == 'object':

        categorical_column = c
        break

# ------------------------------------------------------------
# CATEGORY DISTRIBUTION GRAPH
# ------------------------------------------------------------

if categorical_column:

    chart_data = pdf[categorical_column] \
        .astype(str) \
        .value_counts() \
        .head(10)

    plt.figure(figsize=(10, 5))

    chart_data.plot(kind='bar')

    plt.xlabel(categorical_column)

    plt.ylabel("Count")

    plt.title(
        f"Top 10 {categorical_column} Distribution"
    )

    plt.xticks(rotation=45)

    plt.tight_layout()

    category_graph = "category_distribution.png"

    plt.savefig(category_graph)

    plt.close()

    graph_paths.append(category_graph)

# ------------------------------------------------------------
# NULL VALUE ANALYSIS GRAPH
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.bar(
    null_df["Column"],
    null_df["Null_Count"]
)

plt.xlabel("Columns")

plt.ylabel("Null Count")

plt.title("Null Value Analysis")

plt.xticks(rotation=90)

plt.tight_layout()

null_graph = "null_analysis.png"

plt.savefig(null_graph)

plt.close()

graph_paths.append(null_graph)

print("✅ Graphs Generated Successfully")

# ============================================================
# CREATE PDF REPORT
# ============================================================

print("================================================")
print("GENERATING PDF REPORT")
print("================================================")

pdf_report_path = "Swiggy_Zomato_Data_Quality_Report.pdf"

doc = SimpleDocTemplate(
    pdf_report_path,
    pagesize=letter
)

styles = getSampleStyleSheet()

elements = []

# ============================================================
# PDF TITLE
# ============================================================

title = Paragraph(
    "Swiggy vs Zomato Data Quality Report",
    styles['Title']
)

elements.append(title)

elements.append(Spacer(1, 20))

# ============================================================
# PROJECT SUMMARY
# ============================================================

summary_text = f"""
<b>Total Rows:</b> {row_count}<br/>
<b>Total Columns:</b> {column_count}<br/>
<b>Duplicate Rows:</b> {duplicate_count}
"""

summary = Paragraph(
    summary_text,
    styles['BodyText']
)

elements.append(summary)

elements.append(Spacer(1, 20))

# ============================================================
# VALIDATION RESULTS TABLE
# ============================================================

elements.append(
    Paragraph(
        "Data Quality Validation Results",
        styles['Heading2']
    )
)

table_data = [
    ["Validation", "Status"]
]

for row in validation_results:

    table_data.append(row)

validation_table = Table(
    table_data,
    colWidths=[320, 150]
)

validation_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),

    ('GRID', (0, 0), (-1, -1), 1, colors.black),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('BOTTOMPADDING', (0, 0), (-1, 0), 12)

]))

elements.append(validation_table)

elements.append(Spacer(1, 20))

# ============================================================
# NULL VALUE TABLE
# ============================================================

elements.append(
    Paragraph(
        "Null Value Analysis",
        styles['Heading2']
    )
)

null_table_data = [
    ["Column", "Null Count"]
]

for index, row in null_df.iterrows():

    null_table_data.append([
        row["Column"],
        int(row["Null_Count"])
    ])

null_table = Table(
    null_table_data,
    colWidths=[320, 150]
)

null_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.grey),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),

    ('GRID', (0, 0), (-1, -1), 1, colors.black),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold')

]))

elements.append(null_table)

elements.append(Spacer(1, 20))

# ============================================================
# ADD VISUALIZATION GRAPHS TO PDF
# ============================================================

elements.append(
    Paragraph(
        "Visualization Graphs",
        styles['Heading2']
    )
)

for graph in graph_paths:

    if os.path.exists(graph):

        img = Image(
            graph,
            width=450,
            height=250
        )

        elements.append(img)

        elements.append(Spacer(1, 20))

# ============================================================
# BUILD PDF REPORT
# ============================================================

doc.build(elements)

print("✅ PDF Report Generated Successfully")

# ============================================================
# EXPORT CLEANED DATA
# ============================================================

clean_output_path = "cleaned_output"

df.dropDuplicates() \
  .write \
  .mode("overwrite") \
  .csv(clean_output_path, header=True)

print("✅ Cleaned Data Exported")

# ============================================================
# STOP SPARK SESSION
# ============================================================

spark.stop()

print("================================================")
print("PROJECT COMPLETED SUCCESSFULLY")
print("================================================")

✅ Spark Session Created
✅ CSV File Loaded Successfully
DATASET SUMMARY
Total Rows    : 2500
Total Columns : 41
SAMPLE DATA
+-------------+---------------+------+------------------+---------------+-------------+----------------------------+-------------------+-------------------+----------------+-------------+--------------------+-------------+--------------------+-----------------------------+-----------------------+--------------+-----------------------+-----------------------+--------------------------------+--------------------------------+------------------------------+------------------------------+-----------------------------+-----------------------------+-------------------------------+-------------------------------+------------------------------------+------------------------------------+-------------------------------+-------------------------------+-----------------------+-----------------------+---------------------------+--------------------+---------------+--------------